## Setup

Keep `example_data.py`, `data_fidelity_metrics.py`, and `missingness_fidelity_metrics.py` beside this notebook.

In [1]:
#| label: setup
import pandas as pd
from IPython.display import display

from example_data import make_dependence_example
from missingness_fidelity_metrics import (
    missingness_dependence_details,
    missingness_dependence_gap,
)

## 1. Metric and inputs

| Property | Value |
|---|---|
| Metric | Missingness–value dependence gap |
| Family | Missingness fidelity |
| Origin | Devised in the source study |
| Classifier | Predicts $M_j$ as an instrument |
| Input | Original $X,Y$ |
| Holdout | Independent $H^{\mathrm{miss}}$ |
| Range | $[-1,1]$; zero means equal predictive dependence |

Marginal and joint Hellinger distances compare missingness frequencies. This metric asks whether missingness occurs in relation to the **values of the other columns**.

## 2. Mathematics

For a partly missing column $j$, remove that column from the predictors and use its mask column as the binary target:

$$
M_j=\mathbf1\{Z_j\text{ is missing}\},\qquad Z_{-j}=\text{all other columns}.
$$

Fit one classifier on $X$ and another on $Y$:

$$
f_j^X:X_{-j}\rightarrow M_{X,j},\qquad
f_j^Y:Y_{-j}\rightarrow M_{Y,j}.
$$

Evaluate both on the same unseen missingness holdout $H^{\mathrm{miss}}$. Let

$$
a_j^X=\operatorname{AUC}(f_j^X;H^{\mathrm{miss}}),\qquad
a_j^Y=\operatorname{AUC}(f_j^Y;H^{\mathrm{miss}}).
$$

For the eligible-column set $J$, the metric is

$$
D_{\mathrm{dep}}(X,Y;H^{\mathrm{miss}})=
\frac1{|J|}\sum_{j\in J}(a_j^X-a_j^Y).
$$

Each eligible column needs at least five missing and five observed cases in both $X$ and $H^{\mathrm{miss}}$. If $Y$ has no variation in $M_j$, its classifier receives AUC $0.5$.

### Classifier used by the implementation

The source function uses a histogram gradient-boosted tree. Numerical predictors receive median filling plus missingness indicators; categorical predictors receive an explicit missing label and one-hot encoding. The model code remains in `missingness_fidelity_metrics.py`.

## 3. A model-sized companion example

The tiny tables used by the direct metrics cannot satisfy the required five missing and five observed cases. `example_data.py` therefore supplies a deterministic companion sample with the same $X,Y,H^{\mathrm{miss}}$ notation and schema. In $X$ and the holdout, missing `income_k` depends on `age` and `region`; in $Y$, its placement is independent of those values.

In [2]:
#| label: dependence-example

example = make_dependence_example()
X, Y, H_miss = example.X, example.Y, example.H_miss

summary = pd.DataFrame(
    {
        "rows": [len(X), len(Y), len(H_miss)],
        "P(income_k missing)": [
            X["income_k"].isna().mean(),
            Y["income_k"].isna().mean(),
            H_miss["income_k"].isna().mean(),
        ],
    },
    index=["X", "Y", "H_miss"],
)
display(summary.round(4))

,rows,P(income_k missing)
X,180,0.3444
Y,180,0.3556
H_miss,180,0.3833


In [3]:
#| label: dependence-gap

details = missingness_dependence_details(
    X, Y, H_miss, categorical_columns=["region", "outcome"], random_state=42
)
D_dep = missingness_dependence_gap(
    X, Y, H_miss, categorical_columns=["region", "outcome"], random_state=42
)

display(details.round(4))
print(f"Missingness–value dependence gap = {D_dep:.6f}")
assert abs(D_dep - details["gap"].mean()) < 1e-12

,column,AUC_X,AUC_Y,gap
0,income_k,0.9001,0.4347,0.4654


Missingness–value dependence gap = 0.465400


## 4. Interpretation

The positive gap means that $X$ teaches the holdout's missingness mechanism better than $Y$ does: $Y$ has lost missingness–value dependence. A value near zero means equal predictive dependence, while a negative value means $Y$ teaches the holdout mechanism better than $X$.

This is a directional gap, not a symmetric distance: swapping $X$ and $Y$ changes its sign and also changes which missingness process defines the evaluation target.

## Takeaway

Train one missingness classifier per eligible column on each table, score both on unseen rows from the reference missingness process, and average $\mathrm{AUC}_X-\mathrm{AUC}_Y$.